# OpenPlaque — Secondary Branch Lateral Divergence

Freeze the validated LAD/takeoff/proximal-trunk geometry and test only whether a second coronary-sized trajectory diverges progressively away from that reference. No global LAD search and no automatic LCX label. Research use only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls — True + valid cache reuses; False forces recomputation/overwrite.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_BRANCH_SEARCH = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install dependencies


In [ ]:
%pip -q install scipy matplotlib pandas
print('Dependencies ready.')


## Step 4 — Load this fresh branch


In [ ]:
import os, sys, subprocess, shutil
REPO='/content/OpenPlaque'
BRANCH='secondary-branch-lateral-divergence-from-main'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',REPO],check=True)
sys.path.insert(0,os.path.join(REPO,'src'))
print('Loaded',BRANCH)


## Step 5 — Initialize and inspect caches


In [ ]:
from openplaque.secondary_branch_lateral_divergence import SecondaryBranchLateralDivergenceWorkflow
reuse={
 'source_ct':REUSE_SOURCE_CT,
 'frozen_geometry':REUSE_FROZEN_GEOMETRY,
 'branch_search':REUSE_BRANCH_SEARCH,
 'figures':REUSE_FIGURES,
 'report':REUSE_REPORT,
}
wf=SecondaryBranchLateralDivergenceWorkflow(reuse=reuse)
display(wf.cache_status())


## Step 6 — Freeze established anatomy

Reuse the disk-backed source CCTA, alternative 1 LAD/root reference, the prior takeoff candidate, the passed 8.8-mm proximal trunk, and the validated RCA calibration. Nothing is rediscovered here.


In [ ]:
wf.load_source_ct()
info=wf.load_frozen_geometry()
print('Frozen takeoff candidate:',info)


## Step 7 — Search specifically for monotonic lateral divergence

The beam search strongly favors trajectories whose distance from the established LAD/root reference increases with arc length while serial source-resolution lumen quality remains coronary-like. A trajectory may stay close immediately after the candidate, but after ~1–3 mm it is penalized for turning back toward the reference.


In [ ]:
summary=wf.search_branch(max_length_mm=11.0,beam_width=28)
print(summary)
display(wf.candidates.head(20))
display(wf.branch_qc)


## Step 8 — Generate decisive QC figures


In [ ]:
wf.make_figures()
from IPython.display import Image, display
for fn in ['01_lateral_branch_cross_sections.png','02_branch_vs_reference_mips.png','03_monotonic_separation_profile.png','04_candidate_leaderboard.png']:
    print(fn)
    display(Image(filename=str(wf.out/fn)))


## Step 9 — Build report-back ZIP

PASS requires at least 6 mm of coronary-like persistence, strong serial-plane QC, at least 3 mm endpoint separation from the LAD/root reference, at least 2 mm net separation gain, and predominantly monotonic divergence. PASS supports an independent coronary-like secondary branch only; it is not automatically labeled LCX.


In [ ]:
zip_path=wf.make_report()
print('REPORT_BACK:',zip_path)
print('Expected filename: OPENPLAQUE_SECONDARY_BRANCH_LATERAL_DIVERGENCE_REPORT_BACK.zip')
